# SO3C GPU port validation (internal protocol)

Blocking gate. GPU float32 must land inside the CPU float64 reference band
before any canonical scaling run is worth its GPU-hours.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import subprocess, sys, time

def ensure_torch_for_this_gpu():
    import torch
    if not torch.cuda.is_available():
        print("no CUDA; nothing to fix")
        return
    cap = torch.cuda.get_device_capability(0)
    tag = "sm_%d%d" % cap
    if tag in torch.cuda.get_arch_list():
        print("torch", torch.__version__, "already supports", tag)
        return
    print("torch", torch.__version__, "lacks", tag,
          "- installing a compatible build")
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.5.1", "--index-url",
                        "https://download.pytorch.org/whl/cu121"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install a GPU-compatible torch")

ensure_torch_for_this_gpu()
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"],
               check=True)

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import glob, pathlib
# The data-prep kernel wrote data/top_tagging_*.npz, so the mounted path is
# one level deeper than /kaggle/input/<source>/ -- search recursively and
# show what is actually mounted when nothing matches.
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
if not cands:
    for root in sorted(glob.glob("/kaggle/input/*")):
        print("mounted:", root)
        for sub in sorted(glob.glob(root + "/**/*", recursive=True))[:20]:
            print("   ", sub)
    raise AssertionError("top_tagging_train.npz not found under /kaggle/input")
DATA = str(pathlib.Path(cands[0]).parent)
print("data:", DATA)
REF = {"so3c_equivariant_set": (0.9710, 0.0011),
       "so3c_invariant_set":   (0.9626, 0.0005),
       "eta_invariants":       (0.9447, 0.0004)}

In [ ]:
# Run the same protocol twice, in float64 and in float32. The CPU reference
# is float64, so comparing a float32 GPU run against it confounds two
# changes at once: hardware (different reduction order) and precision. The
# float64 GPU run isolates the first; the float32-vs-float64 gap on the
# same card measures the second.
for dt in ("float64", "float32"):
    ok = run(["benchmarks.run_top_tagging",
              "--cache-dir", DATA, "--representation", "constituents",
              "--max-samples", "100000", "--epochs", "30",
              "--normalize", "global", "--seed", "0",
              "--device", "cuda", "--dtype", dt, "--batch-size", "512",
              "--models", ",".join(REF),
              "--results-dir", "/kaggle/working/results_" + dt,
              "--ckpt-dir", "/kaggle/working/checkpoints/" + dt, "--resume",
              "--max-seconds", "36000"])
    assert ok, dt + " run failed"

In [ ]:
import json

def load(dt, m):
    p = "/kaggle/working/results_%s/top_tagging_constituents__%s__seed0.json"
    return json.load(open(p % (dt, m)))["test_metrics"]

print("%-24s%10s%10s%10s%12s" % ("model", "GPU f64", "GPU f32",
                                 "CPU f64", "f64-CPU"))
worst_hw = 0.0
worst_prec = 0.0
for m, (ref, sd) in REF.items():
    a64 = load("float64", m)["test_auc"]
    a32 = load("float32", m)["test_auc"]
    worst_hw = max(worst_hw, abs(a64 - ref))
    worst_prec = max(worst_prec, abs(a32 - a64))
    print("%-24s%10.4f%10.4f%10.4f%+12.4f" % (m, a64, a32, ref, a64 - ref))

print()
print("hardware/reduction-order effect (GPU f64 vs CPU f64): %.4f AUC"
      % worst_hw)
print("precision effect (f32 vs f64 on the same card):        %.4f AUC"
      % worst_prec)
assert worst_hw < 0.002, "GPU float64 does not reproduce the CPU reference"
print()
print("PORT VALIDATED - the float32 offset is precision, not a port bug"
      if worst_prec < 0.005 else
      "WARNING: float32 costs more than 0.005 AUC; run the campaign in f64")

In [ ]:
# Checkpoint/resume on CUDA: interrupt, resume, compare per-epoch history.
import json, shutil, pathlib
shutil.rmtree("/kaggle/working/rc", ignore_errors=True)
base = ["benchmarks.run_top_tagging", "--cache-dir", DATA,
        "--representation", "constituents", "--max-samples", "20000",
        "--epochs", "6", "--normalize", "global", "--seed", "0",
        "--device", "cuda", "--dtype", "float32", "--batch-size", "256",
        "--models", "so3c_invariant_set"]
run(base + ["--results-dir", "/kaggle/working/rc/cont"])
run(base + ["--results-dir", "/kaggle/working/rc/int",
            "--ckpt-dir", "/kaggle/working/rc/ck", "--max-seconds", "20"])
for f in pathlib.Path("/kaggle/working/rc/int").glob("*.json"):
    f.unlink()
run(base + ["--results-dir", "/kaggle/working/rc/int",
            "--ckpt-dir", "/kaggle/working/rc/ck", "--resume"])
name = "top_tagging_constituents__so3c_invariant_set__seed0.json"
c = json.load(open("/kaggle/working/rc/cont/" + name))
i = json.load(open("/kaggle/working/rc/int/" + name))
same = ([h["val_acc"] for h in c["history"]] ==
        [h["val_acc"] for h in i["history"]])
print("resume reproduces continuous training on CUDA:", same)
print("AUC %.10f vs %.10f" % (c["test_metrics"]["test_auc"],
                              i["test_metrics"]["test_auc"]))